# Lab 04 — TF-IDF

**Curso:** Diplomado en Inteligencia Artificial  
**Nivel:** Introductorio–Intermedio  
**Duración estimada:** 45–60 minutos  
**Prerequisito:** Lab 03 — Bag of Words

---

## Contexto — El problema de BoW

En el Lab anterior construiste una matriz BoW sobre el corpus de Wikipedia.  
Las palabras con mayor frecuencia total eran: *de, la, el, en, que, se, un…*

Estas palabras aparecen en **todos** los documentos. BoW les asigna conteos altísimos  
y eso hace que dominen cualquier comparación. Dos artículos sobre temas distintos  
parecen similares porque comparten 
, 
 y 
.

**La pregunta clave:**
> ¿Cómo diferenciamos una palabra informativa (como `"Pirineos"`) de una que no aporta nada (`"de"`)?

La respuesta es **TF-IDF**.

> 📌 TF-IDF no es una alternativa a BoW — es una **mejora**.  
> Sigue siendo una bolsa de palabras, pero los conteos se reemplazan por pesos  
> que reflejan la importancia real de cada término en cada documento.

In [ ]:
# ⚙️ Configuración — Ejecuta esta celda primero
!pip install scikit-learn pandas --quiet

import os
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("✅ Librerías cargadas correctamente")

---

## ⚙️ Carga del Corpus

Usaremos el mismo `eswiki_corpus.txt` del Lab anterior.

> ⚠️ **Si estás en Google Colab:** ejecuta la siguiente celda para subir el archivo o montarlo desde Drive.  
> Si estás en local (VS Code / Jupyter), puedes saltar esa celda — el archivo se lee directamente.

In [ ]:
import os

EN_COLAB = 'google.colab' in str(get_ipython())

if EN_COLAB:
    # --- OPCIÓN A1: Montar Google Drive ---
    # Sube eswiki_corpus.txt a tu Drive (ej. en /Mi unidad/PLN/)
    # y ajusta la ruta CORPUS_FILE abajo.

    # from google.colab import drive
    # drive.mount('/content/drive')
    # CORPUS_FILE = '/content/drive/MyDrive/PLN/eswiki_corpus.txt'

    # --- OPCIÓN A2: Subir el archivo directamente ---
    from google.colab import files
    print("Sube el archivo eswiki_corpus.txt cuando aparezca el selector:")
    uploaded = files.upload()
    CORPUS_FILE = list(uploaded.keys())[0]

else:
    CORPUS_FILE = os.path.join(os.path.dirname(os.path.abspath('__file__')),
                               '..', 'eswiki_corpus.txt')

print(f"✅ Ruta del corpus: {CORPUS_FILE}")
print(f"   Existe: {os.path.exists(CORPUS_FILE)}")

In [ ]:
# ⚙️ Función de limpieza + carga del corpus
# Ejecuta esta celda antes de continuar

def limpiar(texto):
    """Elimina ruido típico de un dump de Wikipedia en español."""
    texto = re.sub(r'&lt;.*?&gt;', ' ', texto)
    texto = re.sub(r'<[^>]+>', ' ', texto)
    texto = re.sub(r'\[\[.*?\]\]', ' ', texto)
    texto = re.sub(r'\{\{.*?\}\}', ' ', texto)
    texto = re.sub(r"'{2,}", '', texto)
    texto = re.sub(r'&\w+;', ' ', texto)
    texto = re.sub(r'[^a-záéíóúüñA-ZÁÉÍÓÚÜÑ\s]', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto).strip().lower()
    return texto

STOPWORDS_ES = [
    'a', 'al', 'algo', 'algunas', 'algunos', 'ante', 'antes', 'como',
    'con', 'contra', 'cual', 'cuando', 'de', 'del', 'desde', 'donde',
    'durante', 'e', 'el', 'ella', 'ellas', 'ellos', 'en', 'entre',
    'era', 'es', 'esa', 'esas', 'ese', 'eso', 'esos', 'esta', 'estas',
    'este', 'estos', 'fue', 'ha', 'hay', 'he', 'la', 'las', 'le', 'les',
    'lo', 'los', 'mas', 'me', 'mi', 'muy', 'ni', 'no', 'nos', 'o', 'os',
    'para', 'pero', 'por', 'que', 'se', 'si', 'sin', 'sobre', 'son',
    'su', 'sus', 'tambien', 'tanto', 'te', 'tiene', 'tu', 'tus',
    'un', 'una', 'uno', 'unos', 'unas', 'y', 'ya', 'yo', 'ser', 'han',
    'sido', 'parte', 'dicho', 'tras',
]

N_LINEAS = 50   # Cambia este número para usar más oraciones

with open(CORPUS_FILE, 'r', encoding='utf-8') as f:
    lineas_raw = [l.strip() for l in f if l.strip()]

CORPUS_RAW = lineas_raw[:N_LINEAS]
oraciones_limpias = [limpiar(o) for o in CORPUS_RAW]
oraciones_limpias = [o for o in oraciones_limpias if len(o.split()) > 3]

print(f"✅ Corpus cargado: {len(CORPUS_RAW)} líneas leídas → {len(oraciones_limpias)} oraciones válidas")
print(f"\nEjemplo:")
print(f"  ANTES:   {CORPUS_RAW[2][:120]}")
print(f"  DESPUÉS: {oraciones_limpias[2][:120]}")

---

## Parte 1 — ¿Por qué BoW falla?

### 📝 Ejercicio 1 — Visualiza el problema

Descomenta el bloque y ejecuta. Observa el top-10 de palabras por frecuencia BoW  
sobre las oraciones del corpus.

> ¿Alguna de esas palabras dice algo útil sobre el **tema** de los documentos?

> ¿Qué pasaría si usaras esas palabras para comparar un artículo sobre Andorra  
> con uno sobre la Revolución Francesa?

In [ ]:
# Ejercicio 1 — Descomenta y ejecuta

# bow_raw  = CountVectorizer(max_features=500)
# X_raw    = bow_raw.fit_transform(oraciones_limpias)
# conteo   = X_raw.toarray().sum(axis=0)
# palabras = bow_raw.get_feature_names_out()
# top10    = sorted(zip(palabras, conteo), key=lambda x: -x[1])[:10]

# print("Top-10 palabras más frecuentes (BoW sin filtrar):")
# for i, (p, f) in enumerate(top10, 1):
#     print(f"  {i:2}. {p:<20} {int(f):>5} apariciones")

---

## Parte 2 — Cálculo Manual de TF-IDF

Mismo mini-corpus del Lab anterior:

| Doc | Texto |
|-----|-------|
| D1  | `andorra es un estado pequeño` |
| D2  | `andorra tiene capital en andorra` |
| D3  | `españa es un estado europeo` |

---

### Paso 2a — TF (Term Frequency)

$$TF(t, d) = \frac{\text{veces que aparece } t \text{ en } d}{\text{total de palabras en } d}$$

### 📝 Ejercicio 2a — Calcula el TF para D1 (5 palabras en total)

| Palabra  | Conteo en D1 | TF = conteo / 5 |
|----------|--------------|-----------------|
| andorra  | 1            | 1/5 = ___       |
| es       | 1            | 1/5 = ___       |
| estado   | 1            | 1/5 = ___       |
| pequeño  | 1            | 1/5 = ___       |
| un       | 1            | 1/5 = ___       |

*(completa las celdas `___`)*

---

### Paso 2b — IDF (Inverse Document Frequency)

$$IDF(t) = \ln\left(\frac{N}{df_t}\right)$$

Donde **N** = total de documentos y **df_t** = documentos que contienen el término.

> Si una palabra aparece en **todos** los documentos → IDF = 0 (no distingue nada).  
> Si aparece en **solo uno** → IDF máximo (muy específica).

### 📝 Ejercicio 2b — Completa la tabla IDF (N = 3 documentos)

| Palabra  | ¿En D1? | ¿En D2? | ¿En D3? | df | IDF = ln(3/df) |
|----------|---------|---------|---------|----|----------------|
| andorra  | Sí      | Sí      | No      | 2  | ln(3/2) = ___  |
| capital  | No      | Sí      | No      | 1  | ln(3/1) = ___  |
| es       | Sí      | No      | Sí      | 2  | ln(3/2) = ___  |
| estado   | Sí      | No      | Sí      | 2  | ln(3/2) = ___  |
| europeo  | No      | No      | Sí      | 1  | ln(3/1) = ___  |
| pequeño  | Sí      | No      | No      | 1  | ln(3/1) = ___  |
| tiene    | No      | Sí      | No      | 1  | ln(3/1) = ___  |
| un       | Sí      | No      | Sí      | 2  | ln(3/2) = ___  |

*Usa: ln(1.5) ≈ 0.41 y ln(3) ≈ 1.10*

---

### Paso 2c — TF-IDF = TF × IDF

### 📝 Ejercicio 2c — TF-IDF para D1

Con TF = 0.20 para todas las palabras y los IDF calculados arriba:

| Palabra | TF   | IDF  | TF-IDF = TF × IDF |
|---------|------|------|-------------------|
| andorra | 0.20 | 0.41 | ___               |
| es      | 0.20 | 0.41 | ___               |
| estado  | 0.20 | 0.41 | ___               |
| pequeño | 0.20 | 1.10 | ___               |
| un      | 0.20 | 0.41 | ___               |

> ¿Cuál es la palabra más característica de D1 según TF-IDF? ¿Tiene sentido?

*(Escribe tu respuesta aquí)*

In [ ]:
# Ejercicio 2 — Verifica tus cálculos manuales con sklearn
# Descomenta y ejecuta

# mini_corpus = [
#     "andorra es un estado pequeño",
#     "andorra tiene capital en andorra",
#     "españa es un estado europeo",
# ]

# tfidf_mini = TfidfVectorizer(use_idf=True, smooth_idf=False, norm=None)
# X_mini     = tfidf_mini.fit_transform(mini_corpus)
# df_mini    = pd.DataFrame(
#     X_mini.toarray().round(3),
#     columns=tfidf_mini.get_feature_names_out(),
#     index=["D1", "D2", "D3"]
# )
# print("Matriz TF-IDF del mini-corpus:")
# print(df_mini)
# print()
# print("Nota: sklearn usa suavizado → IDF = ln((N+1)/(df+1)) + 1")
# print("Los valores difieren un poco del cálculo manual, pero el RANKING es el mismo.")
# print()
# print("Palabra más característica por documento:")
# for i, doc in enumerate(["D1", "D2", "D3"]):
#     row = df_mini.iloc[i]
#     print(f"  {doc}: '{row.idxmax()}' (TF-IDF = {row.max():.3f})")

---

## Parte 3 — `TfidfVectorizer` sobre el corpus real

`TfidfVectorizer` aplica automáticamente TF-IDF a todos los documentos del corpus.

### 📝 Ejercicio 3a — Aplicar TF-IDF al corpus

Descomenta y ejecuta.

> ¿La forma de la matriz es la misma que en BoW?  
> ¿Los valores son enteros o decimales? ¿Por qué?

In [ ]:
# Ejercicio 3a — Descomenta y ejecuta

# tfidf   = TfidfVectorizer(max_features=500, stop_words=STOPWORDS_ES)
# X_tfidf = tfidf.fit_transform(oraciones_limpias)

# print(f"Forma de la matriz TF-IDF: {X_tfidf.shape}")
# print(f"  → {X_tfidf.shape[0]} documentos  ×  {X_tfidf.shape[1]} palabras")

### 📝 Ejercicio 3b — Palabras más características por documento

Descomenta y ejecuta. Muestra las top-5 palabras con mayor TF-IDF para los primeros 3 documentos.

> ¿Las palabras son más descriptivas del contenido que las del top-10 BoW de la Parte 1?

In [ ]:
# Ejercicio 3b — Descomenta y ejecuta (requiere haber ejecutado 3a)

# df_tfidf = pd.DataFrame(X_tfidf.toarray(),
#                         columns=tfidf.get_feature_names_out())

# print("Top-5 palabras más características por documento (TF-IDF):")
# print("-" * 60)
# for i in range(min(3, len(df_tfidf))):
#     row  = df_tfidf.iloc[i]
#     top5 = row.nlargest(5)
#     reporte = "  ,  ".join([f"{p} ({v:.3f})" for p, v in top5.items()])
#     print(f"  Doc {i}: {reporte}")

---

## Parte 4 — BoW vs TF-IDF: Comparación Directa

### 📝 Ejercicio 4 — Mismo documento, dos representaciones

Descomenta y ejecuta. Verás las top-10 palabras para el mismo documento  
según BoW y según TF-IDF en columnas paralelas.

> ¿Cuál de las dos representaciones describe mejor el contenido del documento?  

> ¿Qué información perdería un algoritmo de clasificación si usara BoW en lugar de TF-IDF?

> Cambia `DOC_IDX` para ver otros documentos.

In [ ]:
# Ejercicio 4 — Descomenta y ejecuta (requiere haber ejecutado 3a y 3b)

# DOC_IDX  = 0   # ← cambia este número para ver otro documento

# bow_sw   = CountVectorizer(max_features=500, stop_words=STOPWORDS_ES)
# X_bow_sw = bow_sw.fit_transform(oraciones_limpias)
# df_bow_sw = pd.DataFrame(X_bow_sw.toarray(),
#                          columns=bow_sw.get_feature_names_out())

# top_bow   = df_bow_sw.iloc[DOC_IDX].nlargest(10)
# top_tfidf = df_tfidf.iloc[DOC_IDX].nlargest(10)

# print(f"Documento {DOC_IDX}: '{oraciones_limpias[DOC_IDX][:70]}...'")
# print()
# print(f"  {'BoW (conteos)':<25} {'TF-IDF (pesos)'}")
# print(f"  {'-'*24} {'-'*24}")
# for (p_bow, v_bow), (p_tfidf, v_tfidf) in zip(top_bow.items(), top_tfidf.items()):
#     print(f"  {p_bow:<22} {int(v_bow):>3}    {p_tfidf:<22} {v_tfidf:.3f}")

---

## Parte 5 — Similitud Coseno

TF-IDF convierte cada documento en un vector. Podemos medir qué tan similares son  
dos documentos calculando el **ángulo** entre sus vectores:

$$\text{similitud}(d_1, d_2) = \cos(\theta) = \frac{d_1 \cdot d_2}{||d_1|| \cdot ||d_2||}$$

El resultado está entre **0** (completamente distintos) y **1** (idénticos).

### 📝 Ejercicio 5a — Predicción antes de ejecutar

Con el mini-corpus de la Parte 2:
- D1 y D3 comparten `"es"`, `"estado"` y `"un"`
- D1 y D2 solo comparten `"andorra"`
- D2 y D3 no comparten ninguna palabra

> Antes de ejecutar: ¿qué par de documentos esperas que tenga mayor similitud coseno?

*(Escribe tu respuesta aquí)*

### 📝 Ejercicio 5b — Similitud sobre el corpus real

Descomenta y ejecuta. Se muestra la matriz de similitud para los primeros 10 documentos.

> ¿Qué par tiene mayor similitud? ¿Tiene sentido temáticamente?  

> ¿Qué par tiene similitud 0.0? ¿Por qué?

In [ ]:
# Ejercicio 5 — Descomenta y ejecuta (requiere haber ejecutado 3a)

# N_DOCS = min(10, len(oraciones_limpias))
# sim    = cosine_similarity(X_tfidf[:N_DOCS])
# df_sim = pd.DataFrame(
#     sim.round(2),
#     columns=[f"D{i}" for i in range(N_DOCS)],
#     index=[f"D{i}" for i in range(N_DOCS)],
# )

# print(f"Matriz de similitud coseno (primeros {N_DOCS} documentos):")
# print(df_sim)
# print()

# # Par más similar (ignora la diagonal)
# sim_copy = sim.copy()
# np.fill_diagonal(sim_copy, 0)
# i, j = divmod(sim_copy.argmax(), N_DOCS)
# print(f"Par más similar: D{i} y D{j}  (similitud = {sim_copy[i,j]:.3f})")
# print(f"  D{i}: {oraciones_limpias[i][:80]}")
# print(f"  D{j}: {oraciones_limpias[j][:80]}")

---

## Parte 6 (Bonus) — TF-IDF con N-gramas

### 📝 Ejercicio 6 — Bigramas + TF-IDF

Descomenta y ejecuta con `ngram_range=(1, 2)` (unigramas y bigramas).

> ¿Aparecen bigramas entre las palabras más características?  

> ¿Mejora la representación al combinar unigramas y bigramas?

In [ ]:
# Ejercicio 6 (Bonus) — Descomenta y ejecuta

# tfidf_bg   = TfidfVectorizer(ngram_range=(1, 2), max_features=500,
#                              stop_words=STOPWORDS_ES)
# X_tfidf_bg = tfidf_bg.fit_transform(oraciones_limpias)
# df_tfidf_bg = pd.DataFrame(X_tfidf_bg.toarray(),
#                             columns=tfidf_bg.get_feature_names_out())

# print("Top-5 términos más característicos por documento (unigramas + bigramas):")
# print("-" * 60)
# for i in range(min(3, len(df_tfidf_bg))):
#     row  = df_tfidf_bg.iloc[i]
#     top5 = row.nlargest(5)
#     reporte = "  ,  ".join([f"{p} ({v:.3f})" for p, v in top5.items()])
#     print(f"  Doc {i}: {reporte}")
# print()
# print("→ ¿Ves bigramas en el top-5?")

---

## 🤔 Reflexión Final

Responde en las celdas de texto a continuación (doble clic para editar).

---

**1. IDF = 0**

Si una palabra aparece en **todos** los documentos del corpus, su IDF es 0 y su TF-IDF también.  
¿Qué tipo de palabras caen en esta categoría?  
¿Cuándo podría esto ser un problema?

*(Escribe tu respuesta aquí)*

---

**2. Tamaño del corpus importa**

El IDF de `"Andorra"` en un corpus de 3 documentos es muy distinto al de un corpus de 50.000.  
¿Por qué? ¿Mejora o empeora la representación al aumentar el corpus?

*(Escribe tu respuesta aquí)*

---

**3. TF-IDF vs Word2Vec**

TF-IDF sigue siendo una bolsa de palabras.  
`"rey"` y `"monarca"` tienen vectores completamente independientes aunque signifiquen lo mismo.  
¿Qué representación necesitarías para que palabras con significados similares  
produzcan vectores similares?

*(Escribe tu respuesta aquí)*

> La respuesta es **Word2Vec** → continúa en el **Lab 05**.